# Sarashina 2.2 ONNX Export

Export SB Intuitions Sarashina 2.2 models to ONNX for use with the Lenzu runtime.

**Runtime:** Google Colab (T4 / A100 GPU recommended)

<a href="https://colab.research.google.com/github/HidekiAI/lenzu/blob/trunk/notebooks/sarashina_export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install with the rust-accelerated transfer layer
!pip install -U "optimum[onnxruntime-gpu]" transformers accelerate hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
from google.colab import drive
import os

# 1. Mount your 20TB Drive
drive.mount('/content/drive')

# 2. Point the Hugging Face cache to your Drive
# This way, the 8GB is saved PERMANENTLY in your 20TB pool
os.environ["HF_HOME"] = "/content/drive/MyDrive/HF_Cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download

models = ["sbintuitions/sarashina2.2-ocr", "sbintuitions/sarashina2.2-vision-3b"]
for m in models:
    print(f"Downloading {m} directly to Google Drive...")
    snapshot_download(repo_id=m, cache_dir="/content/drive/MyDrive/HF_Cache")
    print(f"{m} is now safely stored in your Drive!")

In [ ]:
import shutil
from google.colab import drive

# 1. Mount Drive immediately so we can warp files there instantly
drive.mount('/content/drive', force_remount=True)
DRIVE_PATH = "/content/drive/MyDrive/Lenzu_Exports"
os.makedirs(DRIVE_PATH, exist_ok=True)

models_to_forge = [
    {
        "id": "sbintuitions/sarashina2.2-ocr",
        "name": "ocr_3b",
        "task": "vision-encoder-decoder"
    },
    {
        "id": "sbintuitions/sarashina2.2-0.5B-instruct-v0.1",
        "name": "mini_500m",
        "task": "text-generation-with-past"
    },
    {
        "id": "sbintuitions/sarashina2.2-vision-3b",
        "name": "base_vision_3b",
        "task": "vision-encoder-decoder"
    }
]

for model in models_to_forge:
    out_dir = f"./{model['name']}_onnx"
    zip_file = f"{model['name']}_export.zip"

    if os.path.exists(f"{DRIVE_PATH}/{zip_file}"):
        print(f"Skipping {model['name']}, already in Drive.")
        continue

    print(f"Forging {model['name']} with task {model['task']}...")

    try:
        # We add --trust-remote-code because Sarashina uses custom layers
        !optimum-cli export onnx \\
            --model "{model['id']}" \\
            --task "{model['task']}" \\
            --trust-remote-code \\
            --device cuda \\
            --dtype fp16 \\
            "{out_dir}"

        shutil.make_archive(model['name'], 'zip', out_dir)
        shutil.move(f"{model['name']}.zip", f"{DRIVE_PATH}/{zip_file}")
        shutil.rmtree(out_dir)
        print(f"{model['name']} SUCCESS.")

    except Exception as e:
        print(f"Error during {model['name']}: {e}")